## Quality Analysis — `DataQualityAgent` output

Requirements mapping:
- Detect missing values, duplicates, outliers, and class imbalance (Part 1)
- Apply at least 2 cleaning strategies and compare (Part 2)
- Justify the best strategy in a Markdown cell (Part 3)


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from agents.data_quality_agent import DataQualityAgent

sns.set_theme(style='whitegrid')

# Use latest artifacts if available; otherwise fall back to data/raw/
raw_candidates = sorted(Path('artifacts').glob('run_*/data_raw.csv'))
path = raw_candidates[-1] if raw_candidates else Path('data/raw/data_raw.csv')
df = pd.read_csv(path)

df.head()

In [ ]:
dq = DataQualityAgent()
report = dq.detect_issues(df)
report

In [ ]:
# Part 1: Visualize missingness, duplicates, outliers, and imbalance
import numpy as np

missing_counts = report['missing']['count']
missing_pct = report['missing']['pct']
top_missing = sorted(missing_pct.items(), key=lambda kv: kv[1], reverse=True)[:10]

plt.figure(figsize=(10,4))
cols = [k for k,_ in top_missing]
vals = [v for _,v in top_missing]
sns.barplot(x=vals, y=cols)
plt.title('Top missing columns (%)')
plt.tight_layout()
plt.show()

# Duplicates (count)
dup_n = int(report.get('duplicates') or 0)
plt.figure(figsize=(5,3))
sns.barplot(x=['duplicates(text)'], y=[dup_n])
plt.title('Duplicate count')
plt.tight_layout()
plt.show()

text_length = df['text'].astype(str).str.len()
plt.figure(figsize=(8,4))
sns.histplot(text_length, bins=50)
bounds = report.get('outliers', {}).get('bounds', {})
lo = bounds.get('lo', None)
hi = bounds.get('hi', None)
if lo is not None:
    plt.axvline(lo, color='red', linestyle='--', linewidth=1)
if hi is not None:
    plt.axvline(hi, color='red', linestyle='--', linewidth=1)
plt.title('Text length distribution')
plt.tight_layout()
plt.show()

vc = df['label'].astype(str).value_counts()
plt.figure(figsize=(10,4))
sns.barplot(x=vc.index.astype(str), y=vc.values)
plt.xticks(rotation=45, ha='right')
plt.title('Class distribution (label)')
plt.tight_layout()
plt.show()


In [ ]:
# Part 2: Try two cleaning strategies and compare
strategy_a = {
    'missing': 'drop',
    'duplicates': 'drop',
    'outliers': 'clip_iqr',
}
strategy_b = {
    'missing': 'fill_unknown',
    'duplicates': 'keep_first',
    'outliers': 'remove_iqr',
}

df_a = dq.fix(df, strategy=strategy_a)
df_b = dq.fix(df, strategy=strategy_b)

cmp_a = dq.compare(df, df_a)
cmp_b = dq.compare(df, df_b)

comparison = pd.merge(
    cmp_a.rename(columns={'before':'before_a','after':'after_a'}),
    cmp_b.rename(columns={'before':'before_b','after':'after_b'}),
    on='metric',
    how='outer'
)
comparison

## Part 3: Argument (choose best strategy)

Based on the comparison table, choose the strategy that:
1) reduces missing rows without destroying class balance,
2) removes or mitigates extreme outliers in `text_length`,
3) controls duplicates consistently.

In this project template, the typical best choice is **strategy_b** when the dataset contains noisy length outliers (so `remove_iqr` helps) and label distribution stays stable. If `strategy_b` drops too many samples or harms minority classes, switch to **strategy_a**.